# Jointures de tables

Ce TP rapproche quatre sources sur les départements français : les transactions immobilières de DVF (2014 à juin 2022), la population estimée par l'INSEE, la table des départements et de leurs régions, et l'ensoleillement moyen. Chaque jointure y cache un piège réel : des codes écrits de deux façons, des noms orthographiés autrement, une table à plusieurs lignes par clé. Une jointure fausse ne lève presque jamais d'erreur : elle ajoute, perd ou duplique des lignes en silence. Chaque exercice se termine donc par une **vérification** (un nombre de lignes, une liste de départements, zéro ligne sans partenaire), celles de la dernière frame du cours : type et forme des clés, cardinalité (`validate`), correspondances (`indicator`), nombre de lignes avant et après.

1. Empiler et juxtaposer : `concat`
2. Joindre sur l'index : `join`
3. Joindre sur un nom : l'ensoleillement
4. Joindre sur plusieurs clés : département et année
5. Clés dupliquées
6. Toutes les combinaisons : `how="cross"` (facultatif)

## Mise en place

Les données viennent de [shuuchuu/datasets](https://github.com/shuuchuu/datasets) :

- `dvf/` : les [demandes de valeurs foncières géolocalisées](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) (Etalab), dans la version nettoyée du dépôt [NyxAether/DVF](https://github.com/NyxAether/DVF), une ligne par transaction ;
- `france-departements/` : les départements, leur nom et leur région ;
- `france-population/` : le classeur des [estimations de population par département](https://www.insee.fr/fr/statistiques/1893198) de l'INSEE ;

et l'ensoleillement par département, de [data.gouv.fr](https://www.data.gouv.fr/fr/datasets/donnees-du-temps-densoleillement-par-departements-en-france/).

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh france-departements
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh france-population
!wget -q https://static.data.gouv.fr/resources/donnees-du-temps-densoleillement-par-departements-en-france/20221207-142648/temps-densoleillement-par-an-par-departement-feuille-1.csv -O ensoleillement.csv

DVF a 10,8 millions de lignes et une trentaine de colonnes : on n'en lit que quatre (`columns=`), ce qui tient en 200 Mo. `annee_mutation`, qui partitionne le fichier, arrive en catégorie : on la convertit en entier, sous le nom `annee`. Les codes de département restent tels que DVF les écrit.

2022 s'arrête en juin, et quelques départements ont des trous certaines années (des transactions non publiées) : les comparaisons d'une année à l'autre restent indicatives.

In [ ]:
import pandas as pd

dvf = pd.read_parquet(
  "dvf/dvf-linearized-2014-2022.parquet",
  columns=["annee_mutation", "code_departement", "nature_mutation", "valeur_fonciere"],
)
# L'index lu, id_mutation (10,8 millions de chaînes), ne sert pas : 650 Mo de moins
dvf = dvf.reset_index(drop=True).rename(columns={"annee_mutation": "annee"})
dvf = dvf.astype({"annee": "int64"})
dvf.info(memory_usage="deep")

La table des départements, indexée par leur code. Un code (de département, postal, de commune) se lit toujours en texte, `dtype=str` : lu comme un nombre, il perd ses zéros de tête (`01` devient `1`), ce qui est arrivé aux codes de DVF. `names` (avec `header=0`) fixe les noms des colonnes au lieu de dépendre de l'en-tête du fichier.

In [ ]:
departements = pd.read_csv(
  "france-departements/departements-france.csv",
  dtype=str,
  header=0,
  names=["code_departement", "nom_departement", "code_region", "nom_region"],
  index_col="code_departement",
)
departements.head()

`lire_population(annee)` lit la feuille d'une année du classeur de l'INSEE, comme au TP du chapitre 1 (une feuille par année, en-tête sur deux lignes, lignes de totaux en bas) : une `Series` de la population au 1er janvier, indexée par le code du département.

In [ ]:
POPULATION = "france-population/estim-pop-dep-sexe-gca-1975-2023.xls"


def lire_population(annee: int) -> pd.Series:
  """Population au 1er janvier de `annee`, par code de département."""
  feuille = pd.read_excel(
    POPULATION,
    sheet_name=str(annee),
    header=None,
    skiprows=5,  # titre, année et en-tête sur deux lignes
    usecols=[0, 7],  # code du département, population totale
    names=["code_departement", "population"],
    dtype={"code_departement": str},
  )
  # Les départements seulement, sans les lignes de totaux ni la source
  departement = feuille["code_departement"].str.fullmatch(r"\d{2,3}|2[AB]", na=False)
  lignes = feuille[departement].set_index("code_departement")
  return lignes["population"].astype("int64")


lire_population(2022).head()

## Partie 1 — Empiler et juxtaposer : `concat`

### 1.1 La population de 2014 à 2022

Construisez `population`, la population de chaque département pour chaque année de DVF (`ANNEES`), en une table longue de trois colonnes : `annee`, `code_departement` et `population`. Empilez les `Series` que renvoie `lire_population` avec `pd.concat`, en un seul appel : `keys` garde l'année de chaque ligne, `names` nomme les niveaux de l'index obtenu, et `verify_integrity` vérifie qu'aucune étiquette n'y est en double. Ramenez ensuite les niveaux de l'index en colonnes avec `reset_index`.

**Vérification** : 909 lignes (101 départements × 9 années), 101 par année. Sans `keys`, `verify_integrity` lève une erreur : pourquoi ?

**Documentation utile**

- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

In [ ]:
ANNEES = list(range(2014, 2023))

# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `[lire_population(annee) for annee in ANNEES]` : la liste des neuf `Series`, à passer à un seul `pd.concat` (jamais un `concat` par tour de boucle, qui recopie tout à chaque tour)
- `keys=ANNEES` : un niveau d'index de plus, l'année de chaque `Series`
- `names=["annee", "code_departement"]` : les noms des deux niveaux, l'année puis l'index de chaque `Series`
- `verify_integrity=True` : sans `keys`, chaque code revient neuf fois dans l'index, d'où `ValueError: Indexes have overlapping values`
- `.reset_index()` : les deux niveaux deviennent les colonnes `annee` et `code_departement`, et la `Series` la colonne `population`
- `population["annee"].value_counts()` compte les lignes de chaque année

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
population = pd.concat(
  [lire_population(annee) for annee in ANNEES],
  keys=...,
  names=...,
  verify_integrity=...,
).reset_index()
print(population.shape)
population["annee"].value_counts()
```

#### Solution

In [ ]:
population = pd.concat(
  [lire_population(annee) for annee in ANNEES],
  keys=ANNEES,
  names=["annee", "code_departement"],
  verify_integrity=True,
).reset_index()
print(population.shape)
print(population["annee"].value_counts().sort_index())
population.head()

### 1.2 Ventes et population côte à côte

Comptez les transactions de 2021 par département (`value_counts` sur `code_departement`), et mettez-les à côté de la population de 2021 avec `pd.concat` sur l'axe des colonnes. `concat` aligne les lignes sur les étiquettes de l'index, pas sur leur position : l'ordre des deux `Series` (par nombre de ventes décroissant pour l'une, par code pour l'autre) n'importe pas. Calculez ensuite le nombre de ventes pour 1 000 habitants.

**Vérification** : 101 lignes, une par département, et des `NaN` dans les seules lignes `57`, `67`, `68` et `976`, des départements absents de DVF (voir la partie 2). Si vous en obtenez davantage, regardez les étiquettes sans partenaire, et corrigez la colonne `code_departement` de `dvf` elle-même : les parties suivantes s'en servent.

**Documentation utile**

- [`pandas.Series.str.zfill`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.zfill.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `dvf.loc[dvf["annee"] == 2021, "code_departement"].value_counts()` : le nombre de transactions de 2021 par code
- `pd.concat([lire_population(2021), ventes_2021], axis=1, keys=["population", "ventes"])` : `keys` nomme les colonnes
- 110 lignes au lieu de 101 : DVF écrit `1`, `2`… `9` là où l'INSEE écrit `01`, `02`… `09`. `.str.zfill(2)` complète à gauche par des zéros jusqu'à deux caractères (`"1"` → `"01"`, `"2A"` et `"971"` inchangés)
- `dvf["code_departement"]` est une catégorie : `.str.zfill(2)` renvoie des chaînes, que `.astype("category")` remet en catégorie (10,8 millions de lignes, une centaine de valeurs distinctes)
- Un `NaN` fait passer les ventes en `float64`
- Pour vérifier : 25,6 ventes pour 1 000 habitants dans l'Ain (`01`), et 43,6 dans la Creuse (`23`), le plus haut

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
dvf["code_departement"] = dvf["code_departement"].str.zfill(...).astype(...)
ventes_2021 = dvf.loc[..., "code_departement"].value_counts()
cote_a_cote = pd.concat([lire_population(2021), ventes_2021], axis=..., keys=...)
cote_a_cote["pour_1000"] = 1000 * ... / ...
print(cote_a_cote.shape)
cote_a_cote[cote_a_cote.isna().any(axis=1)]
```

#### Solution

In [ ]:
# Les codes de DVF au format de l'INSEE : "1" → "01"
dvf["code_departement"] = dvf["code_departement"].str.zfill(2).astype("category")
ventes_2021 = dvf.loc[dvf["annee"] == 2021, "code_departement"].value_counts()
cote_a_cote = pd.concat(
  [lire_population(2021), ventes_2021], axis=1, keys=["population", "ventes"]
)
cote_a_cote["pour_1000"] = 1000 * cote_a_cote["ventes"] / cote_a_cote["population"]
print(cote_a_cote.shape)
print(cote_a_cote[cote_a_cote.isna().any(axis=1)])
cote_a_cote.sort_values("pour_1000", ascending=False).head()

## Partie 2 — Joindre sur l'index : `join`

Pour la suite, `par_dep` résume DVF par département, sur toute la période : nombre de transactions, montant total et valeur médiane d'une transaction.

In [ ]:
par_dep = dvf.groupby("code_departement", observed=True, as_index=False).agg(
  nb_ventes=("valeur_fonciere", "size"),
  montant=("valeur_fonciere", "sum"),
  prix_median=("valeur_fonciere", "median"),
)
par_dep.head()

### 2.1 Les noms des départements et des régions

Dans `avec_noms`, ajoutez à `par_dep` le nom de chaque département et celui de sa région (`nom_departement` et `nom_region`, de `departements`) avec `DataFrame.join` : `departements` est indexée par le code, `par_dep` l'a en colonne. Affichez ensuite le montant total par région, du plus grand au plus petit.

**Vérification** : toujours 97 lignes, aucun `NaN` dans `nom_region`, 17 régions (13 en métropole, 4 outre-mer). L'Île-de-France vient en tête, avec 634 milliards d'euros.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `par_dep.join(departements[["nom_departement", "nom_region"]], on="code_departement")` : la colonne `code_departement` de `par_dep` rencontre l'index de `departements`. `join` fait par défaut une jointure à gauche (`how="left"`) : autant de lignes que `par_dep`
- `avec_noms["nom_region"].isna().sum()` et `avec_noms["nom_region"].nunique()` pour vérifier
- `groupby("nom_region")["montant"].sum().sort_values(ascending=False)` : le montant par région

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
avec_noms = par_dep.join(departements[...], on=...)
print(len(avec_noms), avec_noms["nom_region"].isna().sum())
print(avec_noms["nom_region"].nunique())
avec_noms.groupby(...)["montant"].sum().sort_values(ascending=False)
```

#### Solution

In [ ]:
avec_noms = par_dep.join(
  departements[["nom_departement", "nom_region"]], on="code_departement"
)
print(len(avec_noms), avec_noms["nom_region"].isna().sum())
print(avec_noms["nom_region"].nunique())
avec_noms.groupby("nom_region")["montant"].sum().sort_values(ascending=False)

### 2.2 Les départements sans transaction

Quels départements de `departements` n'ont aucune transaction dans DVF ? Joignez `par_dep` à `departements` avec `merge`, en gardant toutes les lignes de `departements` et en marquant l'origine de chaque ligne (`indicator`), puis affichez les départements qui n'ont pas de partenaire dans `par_dep`.

**Vérification** : 97 lignes `both`, aucune `left_only`, et 4 départements sans transaction : `57`, `67` et `68` (la Moselle et l'Alsace ont leur propre livre foncier, hors de DVF), et Mayotte (`976`).

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `departements.reset_index()` : le code redevient une colonne, à joindre avec `on="code_departement"` (ou bien `left_on="code_departement", right_index=True`)
- `how="right"` : toutes les lignes de la table de droite, `departements`
- `indicator=True` : une colonne `_merge`, qui vaut `both`, `left_only` ou `right_only`
- `complet["_merge"].value_counts()` : 97 `both`, 4 `right_only`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
complet = par_dep.merge(departements.reset_index(), on=..., how=..., indicator=...)
print(complet["_merge"].value_counts())
complet.loc[complet["_merge"] == ..., ["code_departement", "nom_departement"]]
```

#### Solution

In [ ]:
complet = par_dep.merge(
  departements.reset_index(), on="code_departement", how="right", indicator=True
)
print(complet["_merge"].value_counts())
complet.loc[complet["_merge"] == "right_only", ["code_departement", "nom_departement"]]

## Partie 3 — Joindre sur un nom : l'ensoleillement

Le fichier d'ensoleillement donne, pour 93 départements, le nombre moyen de jours de soleil par an. Il n'a pas de code, seulement le nom du département, écrit à sa façon. Le rapprocher de `departements` par position (sa n-ième ligne avec le n-ième département) décalerait tout dès le premier département absent : on le rapproche par le nom.

In [ ]:
soleil = pd.read_csv("ensoleillement.csv")
soleil.head()

### 3.1 Rapprocher les noms

Construisez `ensoleillement`, une table à deux colonnes, `code_departement` et `jours_soleil`, d'une ligne par département de `soleil`.

1. Joignez d'abord `soleil` à `departements` sur les noms tels quels avec `merge`, en gardant toutes les lignes de `soleil` et avec `indicator` : les lignes `left_only` sont les noms qui ne correspondent à aucun département. Comparez-les aux noms de `departements`.
2. Écrivez une fonction `normaliser` qui met un nom sous une forme commune (méthodes `.str`), appliquez-la aux noms des deux tables, et corrigez ce qu'elle ne rattrape pas avec une table de correspondance, un dictionnaire passé à `Series.replace`.
3. Joignez sur les noms normalisés, en demandant à `merge` de vérifier qu'un nom ne correspond qu'à un département et inversement (`validate`).

**Vérification** : 93 lignes `both`, aucune `left_only` ; les 8 départements absents du fichier sont les deux de Corse, la Creuse et les cinq d'outre-mer.

**Documentation utile**

- [`pandas.Series.str.replace`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.replace.html)
- [`pandas.Series.str.lower`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.lower.html)
- [`pandas.Series.replace`](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- Le premier essai : `soleil.merge(departements.reset_index(), left_on="Départements", right_on="nom_departement", how="left", indicator=True)` donne 88 `both` et 5 `left_only`
- Les écarts : un tiret ou une espace (`Alpes de Haute Provence`, `Côtes d'Armor`, `Territoire-de-Belfort`), une apostrophe typographique (`Côte-d’Or`), et une faute (`Eure-et-Loire` pour `Eure-et-Loir`), que seule la table de correspondance rattrape
- `normaliser(noms)` : `.str.replace("’", "'")`, `.str.replace("-", " ")`, puis `.str.lower()`
- `CORRECTIONS = {"Eure-et-Loire": "Eure-et-Loir"}`, appliqué avant la normalisation : `soleil["Départements"].replace(CORRECTIONS)`
- `assign(cle=lambda d: normaliser(d["nom_departement"]))` : une colonne de clé calculée sur la table elle-même. `departements["nom_departement"]`, indexée par les codes, s'alignerait sur l'index `0`, `1`, `2`… de `departements.reset_index()` : que des `NaN`
- `how="outer"` montre les deux côtés sans partenaire ; `validate="one_to_one"` lève une `MergeError` si une clé normalisée revient deux fois d'un même côté
- `rename(columns={"Temps d'enseillement (jours/an)": "jours_soleil"})` : un nom de colonne plus court (et sans la coquille du fichier)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
CORRECTIONS = {"Eure-et-Loire": "Eure-et-Loir"}


def normaliser(noms: pd.Series) -> pd.Series:
  return noms.str.replace(...).str.replace(...).str.lower()


gauche = soleil.assign(cle=lambda d: normaliser(d["Départements"].replace(...)))
droite = departements.reset_index().assign(cle=lambda d: normaliser(...))
jointure = gauche.merge(droite, on=..., how=..., indicator=..., validate=...)
print(jointure["_merge"].value_counts())
ensoleillement = jointure.loc[
  jointure["_merge"] == "both", ["code_departement", "Temps d'enseillement (jours/an)"]
].rename(columns={"Temps d'enseillement (jours/an)": "jours_soleil"})
```

#### Solution

In [ ]:
# 1. Les noms tels quels
essai = soleil.merge(
  departements.reset_index(),
  left_on="Départements",
  right_on="nom_departement",
  how="left",
  indicator=True,
)
print(essai["_merge"].value_counts())
print(essai.loc[essai["_merge"] == "left_only", "Départements"].tolist())

# 2. Une forme commune, et une table de correspondance pour le reste
CORRECTIONS = {"Eure-et-Loire": "Eure-et-Loir"}


def normaliser(noms: pd.Series) -> pd.Series:
  """Les mêmes noms malgré les tirets, les apostrophes et la casse."""
  return noms.str.replace("’", "'").str.replace("-", " ").str.lower()


# 3. La jointure sur les noms normalisés
gauche = soleil.assign(cle=lambda d: normaliser(d["Départements"].replace(CORRECTIONS)))
droite = departements.reset_index().assign(
  cle=lambda d: normaliser(d["nom_departement"])
)
jointure = gauche.merge(
  droite, on="cle", how="outer", indicator=True, validate="one_to_one"
)
print(jointure["_merge"].value_counts())
print(jointure.loc[jointure["_merge"] == "right_only", "nom_departement"].tolist())
ensoleillement = (
  jointure.loc[
    jointure["_merge"] == "both",
    ["code_departement", "Temps d'enseillement (jours/an)"],
  ]
  .rename(columns={"Temps d'enseillement (jours/an)": "jours_soleil"})
  .astype({"jours_soleil": "int64"})
)
ensoleillement.sort_values("jours_soleil", ascending=False).head()

## Partie 4 — Joindre sur plusieurs clés : département et année

### 4.1 Ventes et montant par habitant

Agrégez DVF par département et par année (nombre de transactions et montant total) dans `par_dep_annee`, puis joignez-y `population` sur les deux clés, dans `par_habitant`, en gardant toutes les lignes de `par_dep_annee` et en vérifiant que chaque couple (département, année) ne rencontre qu'une ligne de `population`. Calculez le nombre de ventes pour 1 000 habitants et le montant par habitant, puis affichez les 5 départements où le montant par habitant de 2021 est le plus élevé.

**Vérification** : 873 lignes (97 départements × 9 années), aucun `NaN` dans `population`. En 2021, les Alpes-Maritimes (`06`), Paris (`75`) et les Hauts-de-Seine (`92`) dépassent 11 000 € par habitant.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `cle = ["code_departement", "annee"]` : les colonnes qui identifient une ligne, des deux côtés
- `dvf.groupby(cle, observed=True, as_index=False).agg(nb_ventes=("valeur_fonciere", "size"), montant=("valeur_fonciere", "sum"))` : `observed=True`, parce que `code_departement` est une catégorie
- `merge(population, on=cle, how="left", validate="one_to_one")`
- `.assign(ventes_pour_1000=lambda d: 1000 * d["nb_ventes"] / d["population"], ...)` : des colonnes calculées sur la table jointe, `d`
- Avec `on="code_departement"` seul, chaque ligne rencontrerait les neuf années de son département : neuf fois trop de lignes, et des colonnes `annee_x` et `annee_y`
- `par_habitant[par_habitant["annee"] == 2021].nlargest(5, "montant_par_habitant")`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
cle = ["code_departement", "annee"]
par_dep_annee = dvf.groupby(cle, observed=True, as_index=False).agg(
  nb_ventes=..., montant=...
)
par_habitant = par_dep_annee.merge(population, on=..., how=..., validate=...).assign(
  ventes_pour_1000=lambda d: ...,
  montant_par_habitant=lambda d: ...,
)
print(par_habitant.shape, par_habitant["population"].isna().sum())
par_habitant[par_habitant["annee"] == 2021].nlargest(5, "montant_par_habitant")
```

#### Solution

In [ ]:
cle = ["code_departement", "annee"]
par_dep_annee = dvf.groupby(cle, observed=True, as_index=False).agg(
  nb_ventes=("valeur_fonciere", "size"), montant=("valeur_fonciere", "sum")
)
par_habitant = par_dep_annee.merge(
  population, on=cle, how="left", validate="one_to_one"
).assign(
  ventes_pour_1000=lambda d: 1000 * d["nb_ventes"] / d["population"],
  montant_par_habitant=lambda d: d["montant"] / d["population"],
)
print(par_habitant.shape, par_habitant["population"].isna().sum())
par_habitant[par_habitant["annee"] == 2021].nlargest(5, "montant_par_habitant")

### 4.2 Agréger puis joindre, ou l'inverse

On aurait pu joindre `population` à chacune des 10,8 millions de transactions, puis agréger. Faites-le : joignez `population` à `dvf` sur les deux clés, en vérifiant la cardinalité attendue, puis agrégez le résultat par département et année (nombre de transactions, montant, et population, la même sur toutes les lignes d'un groupe). Comparez la mémoire qu'occupe la table jointe à celle de `par_habitant`, et vérifiez que les deux chemins donnent le même résultat. Supprimez enfin la table jointe (`del`) pour libérer sa mémoire.

**Vérification** : le même résultat (`equals` sur les colonnes communes, dans le même ordre) ; environ 800 Mo pour la table jointe, contre moins de 100 ko pour `par_habitant`.

**Documentation utile**

- [`pandas.DataFrame.memory_usage`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.memory_usage.html)
- [`pandas.DataFrame.equals`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.equals.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `dvf.merge(population, on=cle, how="left", validate="many_to_one")` : plusieurs transactions par couple (département, année) à gauche, une seule ligne de population à droite
- `agg(..., population=("population", "first"))` : la population d'un groupe, prise sur sa première ligne
- `memory_usage(deep=True).sum() / 1e6` : la mémoire en Mo, chaînes comprises
- `resultat.equals(par_habitant[resultat.columns])` : `True` si les valeurs, l'ordre et les types sont les mêmes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
jointe = dvf.merge(population, on=..., how=..., validate=...)
print(f"Table jointe : {jointe.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
print(f"par_habitant : {par_habitant.memory_usage(deep=True).sum() / 1e3:.0f} ko")
resultat = jointe.groupby(cle, as_index=False).agg(
  nb_ventes=..., montant=..., population=...
)
del jointe
resultat.equals(par_habitant[resultat.columns])
```

#### Solution

In [ ]:
jointe = dvf.merge(population, on=cle, how="left", validate="many_to_one")
print(f"Table jointe : {len(jointe)} lignes, ", end="")
print(f"{jointe.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
print(f"par_habitant : {len(par_habitant)} lignes, ", end="")
print(f"{par_habitant.memory_usage(deep=True).sum() / 1e3:.0f} ko")
resultat = jointe.groupby(cle, as_index=False).agg(
  nb_ventes=("valeur_fonciere", "size"),
  montant=("valeur_fonciere", "sum"),
  population=("population", "first"),
)
del jointe
resultat.equals(par_habitant[resultat.columns])

## Partie 5 — Clés dupliquées

### 5.1 Le montant par habitant sur toute la période

Un collègue veut le montant total des transactions de 2014 à 2022 rapporté à la population, par département. Il joint `par_dep` (une ligne par département) à `population` sur `code_departement`. Faites comme lui, et comparez le nombre de lignes et la somme de `montant` avant et après la jointure. Refaites-la en annonçant la forme attendue (`validate`) et lisez l'erreur. Puis corrigez : ramenez `population` à une ligne par département (sa moyenne sur la période) avant de joindre, et calculez le montant par habitant.

**Vérification** : la jointure naïve donne 873 lignes et un montant total neuf fois trop grand ; la jointure corrigée garde les 97 lignes de `par_dep` et la même somme de `montant` (2 077 milliards d'euros).

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `len(...)` et `["montant"].sum()`, sur `par_dep` puis sur `par_dep.merge(population, on="code_departement")`
- Une clé présente 1 fois à gauche et 9 fois à droite donne 1 × 9 = 9 lignes
- `validate="one_to_one"` : `MergeError: Merge keys are not unique in right dataset`. Dans un `try:` / `except pd.errors.MergeError as erreur:`, la cellule continue après l'erreur
- `population.groupby("code_departement", as_index=False)["population"].mean()` : une ligne par département
- `merge(..., how="left", validate="one_to_one")` : autant de lignes que `par_dep`, ou une erreur

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
naive = par_dep.merge(population, on="code_departement")
print(len(par_dep), par_dep["montant"].sum() / 1e9)
print(len(naive), naive["montant"].sum() / 1e9)
try:
  par_dep.merge(population, on="code_departement", validate=...)
except pd.errors.MergeError as erreur:
  print(erreur)

pop_moy = population.groupby(..., as_index=False)["population"].mean()
corrige = par_dep.merge(pop_moy, on=..., how=..., validate=...)
corrige["montant_par_habitant"] = ...
print(len(corrige), corrige["montant"].sum() / 1e9)
```

#### Solution

In [ ]:
naive = par_dep.merge(population, on="code_departement")
print(f"par_dep : {len(par_dep)} lignes, {par_dep['montant'].sum() / 1e9:.0f} Md€")
print(f"naive : {len(naive)} lignes, {naive['montant'].sum() / 1e9:.0f} Md€")
try:
  par_dep.merge(population, on="code_departement", validate="one_to_one")
except pd.errors.MergeError as erreur:
  print(f"MergeError : {erreur}")

pop_moy = population.groupby("code_departement", as_index=False)["population"].mean()
corrige = par_dep.merge(
  pop_moy, on="code_departement", how="left", validate="one_to_one"
)
corrige["montant_par_habitant"] = corrige["montant"] / corrige["population"]
print(f"corrige : {len(corrige)} lignes, {corrige['montant'].sum() / 1e9:.0f} Md€")
corrige.nlargest(5, "montant_par_habitant")

## Partie 6 (facultative) — Toutes les combinaisons : `how="cross"`

### 6.1 Les combinaisons sans transaction

Quelles natures de mutation (`nature_mutation` : vente, échange, adjudication, expropriation…) n'apparaissent jamais dans certains départements ? Construisez la grille de toutes les combinaisons (département de `par_dep`, nature de mutation) par un produit cartésien. Joignez-y le nombre de transactions de chaque combinaison observée dans DVF en marquant l'origine des lignes, et gardez les combinaisons sans partenaire.

**Vérification** : 582 combinaisons (97 × 6), dont 24 sans transaction, toutes des expropriations.

**Documentation utile**

- [`pandas.DataFrame.merge`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `pd.DataFrame({"nature_mutation": dvf["nature_mutation"].cat.categories})` : les 6 natures, en table
- `par_dep[["code_departement"]].merge(natures, how="cross")` : toutes les paires, sans clé
- `dvf.groupby(["code_departement", "nature_mutation"], observed=True, as_index=False).size()` : les combinaisons observées et leur nombre de transactions
- `grille.merge(observees, on=["code_departement", "nature_mutation"], how="left", indicator=True, validate="one_to_one")`, puis les lignes `left_only`
- Remarque : avec deux clés catégorielles, `groupby(..., observed=False)` donne à lui seul toute la grille, avec une taille de 0 pour les combinaisons absentes ; d'où `observed=` toujours explicite

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
natures = pd.DataFrame({"nature_mutation": dvf["nature_mutation"].cat.categories})
grille = par_dep[["code_departement"]].merge(natures, how=...)
cles = ["code_departement", "nature_mutation"]
observees = dvf.groupby(cles, observed=True, as_index=False).size()
complete = grille.merge(observees, on=..., how=..., indicator=..., validate=...)
print(len(grille), complete["_merge"].value_counts())
sans = complete[complete["_merge"] == ...]
sans["nature_mutation"].value_counts()
```

#### Solution

In [ ]:
natures = pd.DataFrame({"nature_mutation": dvf["nature_mutation"].cat.categories})
grille = par_dep[["code_departement"]].merge(natures, how="cross")
cles = ["code_departement", "nature_mutation"]
observees = dvf.groupby(cles, observed=True, as_index=False).size()
complete = grille.merge(
  observees, on=cles, how="left", indicator=True, validate="one_to_one"
)
print(len(grille), complete["_merge"].value_counts().to_dict())
sans = complete[complete["_merge"] == "left_only"]
print(sans["code_departement"].tolist())
sans["nature_mutation"].value_counts()